In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/mohittaneja222@gmail.com/1_dev_food_chain_project/1_Setup/utilities

In [0]:
dbutils.widgets.text("catalog", "goods_ct", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f'/Volumes/goods_ct/bronze/vol_goods/child_company/full_load/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)
print(base_path)


bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_time", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")
print("Total Rows: ", df.count())
df.show(5)


In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("append").saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

In [0]:
# keep only rows where order qty is present

df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# Clean Customer_id -> keep numeric, else set to 999999

df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
        .otherwise("999999")
        .cast("string")
)

# "Tuesday, July 01, 2025" -> "July 01, 2025"

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"),r"^[A-Za-z]+,\s*","")
)

# Parse order_placement_data using multiple possible formats


df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("order_placement_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("order_placement_date"), "MMMM dd, yyyy")
    )
)

# Drop Duplicates

df_orders = df_orders.dropDuplicates(["order_id","order_placement_date","customer_id","order_qty"])

# convert product id to string

df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

#df_orders.display(10)

In [0]:
# check min & max dates

df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
df_orders.display(20)

In [0]:
null_count = df_orders.filter(F.col("order_placement_date").isNull()).count()
total_count = df_orders.count()

print(f"Total rows: {total_count}")
print(f"Null dates: {null_count}")
print(f"Valid dates: {total_count - null_count}")
print(f"Success rate: {((total_count - null_count) / total_count * 100):.2f}%")

In [0]:
df_products = spark.table("goods_ct.silver.products")

df_products.display(20)

In [0]:
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

df_joined.display(20)

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "True").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"),"silver.order_placements = bronze.order_placement_date AND silver.order_id = bronze.order_id And silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


# GOLD



In [0]:
df_gold = spark.sql(f"select order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity from {silver_table};")

df_gold.show(2)

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating new table")
    df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "True").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, silver_table)
    gold_delta.alias("silver").merge(df_gold.alias("gold"),"source.date = gold.date AND source.order_id = gold.order_id And source.product_code = gold.product_code AND source.customer_code = gold.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


#Merge with Parent Company

In [0]:
df_child = spark.sql(f"select order_placement_date as date, product_code, customer_id as customer_code, order_qty as sold_quantity from {gold_table};")
df_child.display(10)

In [0]:
df_child.count()

### First change the date to first day of the month.
### 2025-07-12 --> 2025-07-1
### 2025-07-13 --> 2025-07-1

In [0]:
df_monthly = (
            # Get Month Start
    df_child
        .withColumn("month_start", F.trunc("date", "MM"))
        # Group monthly grain month_start + product_code + customer_code
        .groupBy("month_start", "product_code","customer_code")
        .agg(
            F.sum("sold_quantity").alias("sold_quantity")
        )

        # Rename month_start to date to matc h your target schema
        .withColumnRenamed("month_start", "date")
)

display(df_monthly.limit(10))


In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")

gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"),"parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute( )